# Phase II: Feature Engineering

Using the price and macro data from Phase I to compute the technical indocators for each ETF. Results are saved to `feature_engineer_csv.csv`

In [1]:
#pip install polars
#pip install ml4t-engineer

In [2]:
import polars as pl
from ml4t.engineer import compute_features
import pandas as pd

c:\Users\ideac\AppData\Local\Programs\Python\Python313\Lib\site-packages\ml4t\engineer\features\ml\__init__.py:9: UserWarning: Feature 'cyclical_encode': lookback=0 but has period/window parameter. Consider using lookback='period' or specifying the actual lookback.
  from ml4t.engineer.features.ml.cyclical_encode import *  # noqa: F403


In [3]:
prices = pl.read_csv("../cleaned_prices.csv")
macro = pl.read_csv("../cleaned_macro.csv")

#Note: Previously, the columns in prices and macro did not have a "date" label, and tickers was undefined.
tickers = prices.columns[1:]
prices = prices.with_columns(pl.col(prices.columns[0]).cast(pl.Date).alias("date")).drop(prices.columns[0])
macro  = macro.with_columns(pl.col(macro.columns[0]).cast(pl.Date).alias("date")).drop(macro.columns[0])

In [9]:
feature_list = ["rsi", "macd", "bollinger_bands", "sma"]

all_ticker_features = []

for ticker in tickers:
    df_ticker = prices.select([
        pl.col("date"),
        pl.col(ticker).cast(pl.Float64).alias("close")
    ]).with_columns([
        pl.col("close").alias("high"),
        pl.col("close").alias("low"),
        pl.col("close").alias("open"),
        pl.lit(1.0).alias("volume")
    ])

    features = compute_features(df_ticker, feature_list)

    # 20-month SMA distance (existing)
    features = features.with_columns(
        ((pl.col("close") / pl.col("sma")) - 1).alias("dist_from_sma")
    )

    # 10-month SMA — Faber's signal period
    features = features.with_columns(
        pl.col("close").rolling_mean(window_size=10).alias("sma10")
    )
    features = features.with_columns([
        ((pl.col("close") / pl.col("sma10")) - 1).alias("dist_from_10m_sma"),
        (pl.col("close") > pl.col("sma10")).cast(pl.Int8).alias("above_10m_sma"),
    ]).drop("sma10")

    features = features.rename({
        col: f"{ticker}_{col}" for col in features.columns if col != "date"
    })

    all_ticker_features.append(features)

final_df = all_ticker_features[0]
for next_df in all_ticker_features[1:]:
    final_df = final_df.join(next_df, on="date", how="left")

final_df = final_df.join(macro, on="date", how="left")
final_df = final_df.drop_nulls()

In [10]:
final_df

date,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_bollinger_bands,SPY_macd,SPY_rsi,SPY_sma,SPY_dist_from_sma,SPY_dist_from_10m_sma,SPY_above_10m_sma,EFA_close,EFA_high,EFA_low,EFA_open,EFA_volume,EFA_bollinger_bands,EFA_macd,EFA_rsi,EFA_sma,EFA_dist_from_sma,EFA_dist_from_10m_sma,EFA_above_10m_sma,IEF_close,IEF_high,IEF_low,IEF_open,IEF_volume,IEF_bollinger_bands,IEF_macd,IEF_rsi,IEF_sma,IEF_dist_from_sma,IEF_dist_from_10m_sma,IEF_above_10m_sma,VNQ_close,VNQ_high,VNQ_low,VNQ_open,VNQ_volume,VNQ_bollinger_bands,VNQ_macd,VNQ_rsi,VNQ_sma,VNQ_dist_from_sma,VNQ_dist_from_10m_sma,VNQ_above_10m_sma,DBC_close,DBC_high,DBC_low,DBC_open,DBC_volume,DBC_bollinger_bands,DBC_macd,DBC_rsi,DBC_sma,DBC_dist_from_sma,DBC_dist_from_10m_sma,DBC_above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,TB3MS
date,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64
2007-09-30,108.383331,108.383331,108.383331,108.383331,1.0,"{111.091612,97.2663,83.440988}",NaN,75.091606,97.2663,0.114295,0.048917,1,46.893768,46.893768,46.893768,46.893768,1.0,"{48.426662,40.742856,33.05905}",NaN,82.834171,40.742856,0.150969,0.060478,1,52.976456,52.976456,52.976456,52.976456,1.0,"{53.340075,50.256057,47.172039}",NaN,72.06748,50.256057,0.054131,0.030467,1,32.989182,32.989182,32.989182,32.989182,1.0,"{38.465251,32.316259,26.167266}",NaN,59.080627,32.316259,0.020823,-0.037596,0,22.830553,22.830553,22.830553,22.830553,1.0,"{22.121983,20.241358,18.360734}",NaN,72.026599,20.241358,0.127916,0.098347,1,4.94,208.292,0.62,114.357,3.89
2007-10-31,109.853683,109.853683,109.853683,109.853683,1.0,"{112.558308,98.345065,84.131821}",NaN,76.353775,98.345065,0.117023,0.052398,1,48.886703,48.886703,48.886703,48.886703,1.0,"{49.389792,41.45177,33.513748}",NaN,85.172442,41.45177,0.179363,0.08756,1,53.546986,53.546986,53.546986,53.546986,1.0,"{53.817447,50.490176,47.162906}",NaN,74.210038,50.490176,0.060543,0.03543,1,33.681648,33.681648,33.681648,33.681648,1.0,"{38.439863,32.601693,26.763524}",NaN,60.603147,32.601693,0.033126,-0.014908,0,24.771681,24.771681,24.771681,24.771681,1.0,"{23.128057,20.556759,17.985461}",NaN,77.712931,20.556759,0.205038,0.164789,1,4.76,208.903,0.54,113.9957,3.9
2007-11-30,105.598785,105.598785,105.598785,105.598785,1.0,"{113.109545,99.13824,85.166935}",NaN,65.940728,99.13824,0.065167,0.007,1,47.115208,47.115208,47.115208,47.115208,1.0,"{49.90716,42.002604,34.098048}",NaN,75.34749,42.002604,0.121721,0.036706,1,55.714413,55.714413,55.714413,55.714413,1.0,"{54.728065,50.865843,47.003622}",NaN,80.370166,50.865843,0.095321,0.066176,1,30.491692,30.491692,30.491692,30.491692,1.0,"{38.38613,32.661683,26.937237}",NaN,51.159663,32.661683,-0.066438,-0.089327,0,24.617361,24.617361,24.617361,24.617361,1.0,"{23.854217,20.838678,17.823138}",NaN,76.383582,20.838678,0.18133,0.130253,1,4.49,210.565,0.93,113.9019,3.27
2007-12-31,104.409668,104.409668,104.409668,104.409668,1.0,"{113.425748,99.815282,86.204816}",NaN,63.340858,99.815282,0.046029,-0.009623,0,45.71056,45.71056,45.71056,45.71056,1.0,"{50.214968,42.39674,34.578511}",NaN,68.591151,42.39674,0.078162,-0.002126,0,55.742527,55.742527,55.742527,55.742527,1.0,"{55.417021,51.258032,47.099044}",NaN,80.435445,51.258032,0.087489,0.057787,1,28.841185,28.841185,28.841185,28.841185,1.0,"{38.334587,32.68896,27.043333}",NaN,47.072504,32.68896,-0.117709,-0.11828,0,26.267769,26.267769,26.267769,26.267769,1.0,"{24.954628,21.136524,17.318421}",NaN,80.270582,21.136524,0.242767,0.174784,1,4.24,211.16,0.99,113.9673,3.0
2008-01-31,98.096947,98.096947,98.096947,98.096947,1.0,"{112.865087,100.313542,87.761996}",NaN,51.689497,100.313542,-0.022097,-0.067854,0,42.123604,42.123604,42.123604,42.123604,1.0,"{50.003297,42.683814,35.364331}",NaN,55.022762,42.683814,-0.013125,-0.078017,0,57.61319,57.61319,57.61319,57.61319,1.0,"{56.44

In [11]:
#DataFrame was using the polars module. I converted it into a pandas DataFrame because csv does not work well with nested information.
#Resulting DF may end up looking weird but it should still be usable since the data can be extracted properly regardless
final_df_pandas = final_df.to_pandas()

In [12]:
final_df_pandas

,date,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_bollinger_bands,SPY_macd,SPY_rsi,SPY_sma,...,DBC_rsi,DBC_sma,DBC_dist_from_sma,DBC_dist_from_10m_sma,DBC_above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,TB3MS
0,2007-09-30,108.383331,108.383331,108.383331,108.383331,1.0,"{'upper': 111.09161222568251, 'middle': 97.266...",NaN,75.091606,97.266300,...,72.026599,20.241358,0.127916,0.098347,1,4.94,208.292,0.62,114.3570,3.89
1,2007-10-31,109.853683,109.853683,109.853683,109.853683,1.0,"{'upper': 112.5583078004291, 'middle': 98.3450...",NaN,76.353775,98.345065,...,77.712931,20.556759,0.205038,0.164789,1,4.76,208.903,0.54,113.9957,3.90
2,2007-11-30,105.598785,105.598785,105.598785,105.598785,1.0,"{'upper': 113.1095454049151, 'middle': 99.1382...",NaN,65.940728,99.138240,...,76.383582,20.838678,0.181330,0.130253,1,4.49,210.565,0.93,113.9019,3.27
3,2007-12-31,104.409668,104.409668,104.409668,104.409668,1.0,"{'upper': 113.42574804972952, 'middle': 99.815...",NaN,63.340858,99.815282,...,80.270582,21.136524,0.242767,0.174784,1,4.24,211.160,0.99,113.9673,3.00
4,2008-01-31,98.096947,98.096947,98.096947,98.096947,1.0,"{'upper': 112.86508727027392, 'middle': 100.31...",NaN,51.689497,100.313542,...,81.769784,21.476621,0.258741,0.175284,1,3.94,212.516,1.50,114.2334,2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-07-31,539.458191,539.458191,539.458191,539.458191,1.0,"{'upper': 549.7205652302619, 'middle': 445.743...",38.022123,70.905999,445.743398,...,51.482590,20.950807,-0.010173,-0.009224,0,5.33,313.534,-0.20,102.8887,5.20
203,2024-08-31,552.062988,552.062988,552.062988,552.062988,1.0,"{'upper': 562.1412563391239, 'middle': 455.015...",40.537823,72.445045,455.015773,...,48.648784,20.887187,-0.027824,-0.024488,0,5.33,314.121,0.00,103.1389,5.05
204,2024-09-30,563.658875,563.658875,563.658875,563.658875,1.0,"{'upper': 576.3199066828046, 'middle': 463.715...",42.971871,73.817242,463.715169,...,49.664542,20.821286,-0.017690,-0.015168,0,5.13,314.686,0.15,102.6418,4.72
205,2024-10-31,558.628967,558.628967,558.628967,558.628967,1.0,"{'upper': 585.595996294541, 'middle': 472.6529...",43.987935,72.139102,472.652940,...,51.721409,20.818663,-0.003449,-0.003441,0,4.83,315.454,0.12,102.2805,4.51


In [13]:
final_df_pandas.to_csv('../feature_engineer_csv.csv', index=False)